In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\bhard\.conda\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
87,"""Don't be greedy"" sums up the depth of this mo...",negative
642,"...un-funny and un-entertaining, possibly the ...",negative
582,This movie is up there with the all-time class...,positive
198,"Great acting, great movie. If you are thinking...",positive
186,I hired the DVD yesterday and first of all it ...,negative


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [6]:
import nltk
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhard\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhard\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\bhard\AppData\Roaming\nltk_data...


True

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
87,greedy sum depth movie rest baloney big budget...,negative
642,un funny un entertaining possibly worst movie ...,negative
582,movie time classic music camera shot acting ex...,positive
198,great acting great movie thinking building see...,positive
186,hired dvd yesterday first started bad aspect r...,negative


In [8]:
df['sentiment'].value_counts()

sentiment
negative    264
positive    236
Name: count, dtype: int64

In [9]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
87,greedy sum depth movie rest baloney big budget...,0
642,un funny un entertaining possibly worst movie ...,0
582,movie time classic music camera shot acting ex...,1
198,great acting great movie thinking building see...,1
186,hired dvd yesterday first started bad aspect r...,0


In [11]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [17]:
vectorizer = CountVectorizer(max_features=80)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
import dagshub

mlflow.set_tracking_uri('your url')
dagshub.init(repo_owner='your dagshub name', repo_name='your project name', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-03-10 11:11:19,233 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Rupeshbhardwaj002/Mlops-Capstone-Project-II "HTTP/1.1 200 OK"


Initialized MLflow to track repo "Rupeshbhardwaj002/Mlops-Capstone-Project-II"

2026-03-10 11:11:19,242 - INFO - Initialized MLflow to track repo "Rupeshbhardwaj002/Mlops-Capstone-Project-II"


Repository Rupeshbhardwaj002/Mlops-Capstone-Project-II initialized!

2026-03-10 11:11:19,244 - INFO - Repository Rupeshbhardwaj002/Mlops-Capstone-Project-II initialized!


<Experiment: artifact_location='mlflow-artifacts:/842b67681d2241b58df0cab355c0c6ab', creation_time=1773120878238, experiment_id='0', last_update_time=1773120878238, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [20]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 80)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-10 11:11:23,634 - INFO - Starting MLflow run...
2026-03-10 11:11:24,249 - INFO - Logging preprocessing parameters...
2026-03-10 11:11:25,541 - INFO - Initializing Logistic Regression model...
2026-03-10 11:11:25,541 - INFO - Fitting the model...
2026-03-10 11:11:25,561 - INFO - Model training complete.
2026-03-10 11:11:25,564 - INFO - Logging model parameters...
2026-03-10 11:11:25,954 - INFO - Making predictions...
2026-03-10 11:11:25,955 - INFO - Calculating evaluation metrics...
2026-03-10 11:11:25,963 - INFO - Logging evaluation metrics...
2026-03-10 11:11:27,454 - INFO - Saving and logging the model...
2026/03/10 11:11:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 11:11:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run aged-rat-43 at: https://dagshub.com/Rupeshbhardwaj002/Mlops-Capstone-Project-II.mlflow/#/experiments/0/runs/5a1e7d53e1d147c19fdbb9ee1f7a95fb
🧪 View experiment at: https://dagshub.com/Rupeshbhardwaj002/Mlops-Capstone-Project-II.mlflow/#/experiments/0
